# Fresh Kaggle T4 x2 Run: Fix 3 + Fix 4

Use a fresh Kaggle notebook/session with **Internet ON** and **Accelerator: T4 x2**.

This notebook runs the next code fixes only:

- Fix 3: multi-metric faithfulness over the completed Fix 2 table.
- Fix 4: cross-dataset tau generalization.

Attach the corrected Fix 2 package before setup. Best input file:

- `fix2_t4x2_outputs_FIXED.zip`

The runner uses `roberta-large-mnli` as the second NLI scorer for Fix 3, avoiding Hugging Face custom remote code. It also handles Kaggle's annoying behavior where uploaded zips appear as extracted dataset folders. Setup fails loudly unless Fix 2 imports as exactly `7500` rows.

Expected output files after packaging:

- `/kaggle/working/fix3_4_t4x2_outputs.zip`
- `/kaggle/working/AAA_FIX3_4_T4X2_OUTPUTS.zip`


In [ ]:
import os
import pathlib
import subprocess
import time

WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'rag-hallucination-detection'
REMOTE = 'https://github.com/Saket-Maganti/rag-hallucination-detection.git'

def run(cmd, cwd=None, check=True):
    print('\n>>>', ' '.join(map(str, cmd)), flush=True)
    return subprocess.run(cmd, cwd=cwd, check=check)

print('START', time.ctime(), flush=True)
run(['bash', '-lc', 'pkill -f kaggle_stream_fix3_4_t4x2 || true; pkill -f kaggle_fix3_4_t4x2 || true; pkill -f fix_03_multimetric_faithfulness.py || true; pkill -f fix_04_tau_generalization.py || true; pkill -x ollama || true'], check=False)
run(['nvidia-smi'], check=False)

if not (REPO / '.git').exists():
    run(['git', 'clone', '--progress', '--branch', 'main', REMOTE, str(REPO)], cwd=WORK)
else:
    run(['git', '-C', str(REPO), 'fetch', 'origin', 'main'], check=False)
    run(['git', '-C', str(REPO), 'checkout', 'main'], check=False)
    run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'main'], check=False)

os.chdir(REPO)
run(['git', 'log', '-1', '--oneline'])
run(['bash', '-n', 'scripts/kaggle_fix3_4_t4x2.sh'])
run(['python', '-m', 'py_compile', 'scripts/kaggle_stream_fix3_4_t4x2.py', 'experiments/fix_03_multimetric_faithfulness.py', 'experiments/fix_04_tau_generalization.py'])
print('Repo ready:', REPO, flush=True)


## 0. Find Fix 2 Input

Run this if you are not sure Kaggle can see your corrected Fix 2 output. It searches both zipped and extracted dataset layouts.


In [ ]:
from pathlib import Path
import csv

roots = [Path('/kaggle/input'), Path('/kaggle/working')]
print('--- candidate zips ---')
for root in roots:
    if root.exists():
        for p in root.rglob('*'):
            if p.is_file() and p.suffix.lower() == '.zip' and 'fix2' in p.name.lower():
                print(p, 'MB=', round(p.stat().st_size / 1024 / 1024, 2))

print('--- candidate Fix 2 CSVs ---')
for root in roots:
    if root.exists():
        for p in root.rglob('*.csv'):
            s = str(p)
            if '/fix_02/' not in s:
                continue
            try:
                with p.open(newline='') as f:
                    n = max(0, sum(1 for _ in csv.reader(f)) - 1)
            except Exception as exc:
                n = f'ERR:{exc}'
            print(p, 'rows=', n)


## 1. Setup + Fix 2 Import + Ollama Preflight

This installs dependencies, imports/rebuilds Fix 2 if needed, verifies exactly `7500` Fix 2 rows, installs/verifies Ollama, starts two Ollama servers, pulls `mistral`, and checks both GPU ports. Wait for `preflight OK` before running the next cell.


In [ ]:
!python -u scripts/kaggle_stream_fix3_4_t4x2.py --stage setup --heartbeat 15

## 2. Run Fix 3 + Fix 4 in Parallel

Fix 3 runs on GPU0 / Ollama port `11434`. Fix 4 runs on GPU1 / Ollama port `11435`. Both write partial CSVs, and Fix 4 now resumes from completed dataset/tau partials if interrupted.


In [ ]:
!python -u scripts/kaggle_stream_fix3_4_t4x2.py --stage parallel --heartbeat 30

## 3. Status

Use this after the parallel run, or after an interruption. It only prints row counts and output paths.

In [ ]:
!python -u scripts/kaggle_stream_fix3_4_t4x2.py --stage status --heartbeat 15

## 4. Package Outputs

Run this after the parallel stage ends with `rc=0`. The package cell creates both normal and `AAA_...` zip names.


In [ ]:
!python -u scripts/kaggle_stream_fix3_4_t4x2.py --stage package --heartbeat 15
!ls -lh /kaggle/working/fix3_4_t4x2_outputs.zip /kaggle/working/AAA_FIX3_4_T4X2_OUTPUTS.zip
!find /kaggle/working -maxdepth 2 \( -name '*FIX3_4*.zip' -o -name '*fix3_4*.zip' \) -print

## 5. Direct Download Link

Run this after package succeeds if the Kaggle file browser opens a 404 page.


In [ ]:
from IPython.display import HTML, display
from pathlib import Path
import base64

paths = [
    Path('/kaggle/working/AAA_FIX3_4_T4X2_OUTPUTS.zip'),
    Path('/kaggle/working/fix3_4_t4x2_outputs.zip'),
    Path('/kaggle/working/rag-hallucination-detection/AAA_FIX3_4_T4X2_OUTPUTS.zip'),
    Path('/kaggle/working/rag-hallucination-detection/fix3_4_t4x2_outputs.zip'),
]

p = next(x for x in paths if x.exists())
print('Using:', p, 'size:', round(p.stat().st_size / 1024 / 1024, 2), 'MB')

data = base64.b64encode(p.read_bytes()).decode()
display(HTML(f'''
<a download="fix3_4_t4x2_outputs.zip"
   href="data:application/zip;base64,{data}"
   style="font-size:22px;font-weight:700;color:#0b57d0">
   CLICK HERE TO DOWNLOAD fix3_4_t4x2_outputs.zip
</a>
'''))

## Debug If Anything Fails

Run this only if setup or the parallel run fails.

In [ ]:
!echo '--- git ---'
!git -C /kaggle/working/rag-hallucination-detection log -1 --oneline || true
!echo '--- zips/csvs ---'
!find /kaggle/working /kaggle/input -maxdepth 20 -type f \( -iname '*FIX*.zip' -o -iname '*fix*.zip' -o -iname '*.csv' \) -print -exec ls -lh {} \; 2>/dev/null | head -n 240 || true
!echo '--- rows ---'
!python - <<'PY'
from pathlib import Path
import csv
for base in ['fix_02', 'fix_03', 'fix_04']:
    root = Path('/kaggle/working/rag-hallucination-detection/data/revision') / base
    for p in sorted(root.glob('*.csv')):
        try:
            with p.open(newline='') as f:
                n = max(0, sum(1 for _ in csv.reader(f)) - 1)
        except Exception as exc:
            n = f'ERR:{exc}'
        print(p, n)
PY
!echo '--- processes ---'
!ps aux | grep -E 'ollama|fix3|fix4|kaggle_stream' | grep -v grep || true
!echo '--- gpu ---'
!nvidia-smi || true
!echo '--- wrapper ---'
!tail -n 160 /kaggle/working/fix3_4_t4x2_wrapper.log 2>/dev/null || true
!echo '--- fix3 log ---'
!tail -n 160 /kaggle/working/rag-hallucination-detection/logs/revision/fix_03_kaggle_t4x2.log 2>/dev/null || true
!echo '--- fix4 log ---'
!tail -n 160 /kaggle/working/rag-hallucination-detection/logs/revision/fix_04_kaggle_t4x2.log 2>/dev/null || true
!echo '--- ollama gpu0 ---'
!tail -n 120 /kaggle/working/fix3_4_t4x2_logs/ollama_gpu0.log 2>/dev/null || true
!echo '--- ollama gpu1 ---'
!tail -n 120 /kaggle/working/fix3_4_t4x2_logs/ollama_gpu1.log 2>/dev/null || true
